In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 8.2 MB/s eta 0:00:00


In [2]:
import anthropic
import time
import csv
from tqdm import tqdm
import argparse
from openai import OpenAI

## **CONFIGURATION**

In [3]:
OPENAI_API_KEY    = "OPENAI_API_KEY"
ANTHROPIC_API_KEY = "ANTHROPIC_API_KEY"

MODELS = {
    "gpt-5.4":           "openai",
    "claude-opus-4-6":   "anthropic"
}

# Reflection settings
MAX_REFLECT  = 2     # fixed number of reflection rounds
DELTA        = 0.10  # minimum score change to continue reflecting

openai_client    = OpenAI(
    api_key = OPENAI_API_KEY
)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [4]:
def call_llm(prompt, model):

    provider = MODELS.get(model)

    if provider is None:
      print(f"  Unknown model: {model}")
      return None

    if provider == "openai":
      response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
      return response.choices[0].message.content

    elif provider == "anthropic":
      response = anthropic_client.messages.create(
            model=model,
            temperature=0,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
      )
      return response.content[0].text


    return None

##AUTO-COT STEP GENERATION

In [5]:
def generate_cot_steps_no_gold(model):
    """
    Generate CoT steps for RELATIONSHIP evaluation   WITHOUT gold reference.
    """
    prompt = """You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the RELATIONSHIPS in a PlantUML candidate diagram given
only a problem description.

Cover:

- How to identify expected relationships between classes
  from the problem description
- How to determine whether the RELATIONSHIP ENDPOINTS are correct,
  meaning the relationship connects the correct pair of classes
  required by the problem description
- How to measure Relationship Completeness (0–1):
  fraction of expected relationships present in the candidate
- How to measure Relationship Correctness (0–1):
  fraction of candidate relationships that are fully accurate

A candidate relationship is fully correct only if ALL of the following match the problem description:

1. ENDPOINTS are correct
   - The relationship connects the correct two classes/entities
   - Matching should be based on the intended class pair from the problem description
   - Ignore arrow direction when first matching by class pair, unless direction is essential to the UML relationship type

2. RELATIONSHIP TYPE is correct
   - The type matches the intended semantic relation in the problem description
   - Examples: association, inheritance/generalization, composition, aggregation, dependency, realization

3. MULTIPLICITY / CARDINALITY is correct
   - Multiplicity annotations at both ends should match the intended meaning in the problem description
   - Check both exactness and rationality:
     - If the problem description clearly specifies multiplicity, the candidate should match it exactly
     - If the problem description does not state exact multiplicity, judge whether the candidate cardinalities are reasonable and consistent with the problem definition
   - Cardinalities that contradict the problem description or are semantically implausible should be marked incorrect

A relationship is INCORRECT if any one of the above criteria fails.

Also explain:

- How to match candidate relationships to expected ones by class pair
  (primarily regardless of direction for pairing purposes)
- How to handle extra/hallucinated relationships
  (present in candidate but not supported by the problem description)
- How to handle missing relationships
  (expected from the problem description but absent in candidate)
- How to handle relationships involving unmatched classes
- How to judge partially specified cases conservatively and consistently

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (no gold) using {model}...")
    steps = call_llm(prompt, model)

    with open("cot_steps_relationships_no_gold.txt", "w",
              encoding="utf-8") as f:
        f.write(steps)
    print("  Saved to cot_steps_relationships_no_gold.txt")

    return steps


def generate_cot_steps_with_gold(model):
    """
    Generate CoT steps for RELATIONSHIP evaluation
    WITH gold reference.
    Used by S5 and S6.
    """
    prompt = """You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the RELATIONSHIPS in a PlantUML candidate diagram against
a gold reference diagram, using the problem description
only when the gold diagram is ambiguous or unclear.

Cover:

- How to enumerate all relationships in the gold diagram,
  recording for each:
  (a) the two class endpoints
  (b) the relationship type
      (association, inheritance/generalization,
       composition, aggregation, dependency, realization)

- How to enumerate all relationships in the candidate
  diagram using the same format

- How to match candidate relationships to gold relationships
  by endpoints:
  - Match relationships based on the pair of connected classes
  - Ignore direction initially for matching purposes
  - Treat a relationship as matched if it connects the same
    pair of classes as in the gold diagram

- How to measure Relationship Completeness (0–1):
  number of matched gold relationships / total number of
  gold relationships

- How to measure Relationship Correctness (0–1):
  number of correct candidate relationships / total number
  of candidate relationships

  A candidate relationship is CORRECT only if it is matched
  to a gold relationship and ALL of the following are satisfied:

  1. ENDPOINTS are correct
     - The relationship connects the correct pair of classes
     - If the gold diagram is unclear, refer to the problem
       description to determine the intended endpoints

  2. RELATIONSHIP TYPE is correct
     - association, inheritance/generalization,
       composition, aggregation, dependency, realization

  IMPORTANT:
  - Differences in multiplicity should be ignored for simplicity

  A relationship is INCORRECT if any one of the above
  criteria fails.

  Unmatched candidate relationships count as incorrect.

- How to handle naming variants in class names
  (e.g. OrderMgr vs OrderManager)

- How to handle extra/hallucinated relationships
  (present in candidate but not in gold)

- How to handle missing relationships
  (present in gold but absent in candidate)

- How to handle relationships involving unmatched classes

- How to resolve ambiguity:
  - Use the gold diagram as the primary reference
  - If the gold diagram is ambiguous or incomplete,
    refer to the problem description to determine
    the intended relationships
  - Do not override a clear gold relationship using
    the problem description

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (with gold) using {model}...")
    steps = call_llm(prompt, model)
    with open("cot_steps_relationships_with_gold.txt", "w",
              encoding="utf-8") as f:
        f.write(steps)
    print("  Saved to cot_steps_relationships_with_gold.txt")

    return steps

In [6]:
BackUp1="""You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the RELATIONSHIPS in a PlantUML candidate diagram given
only a problem description. Cover:

- How to identify expected relationships between classes
  from the problem description
- How to measure Relationship Completeness (0-1):
  fraction of expected relationships present in candidate
- How to measure Relationship Correctness (0-1):
  fraction of candidate relationships that are fully accurate
  — meaning ALL TWO of the following match:
  1. Relationship TYPE matches
     (association, inheritance/generalization, composition, aggregation, dependency, realisation)
  2. MULTIPLICITY annotation matches at both ends
     (1, *, 0..1, 1..*, 0..*, etc.)

  A relationship is INCORRECT if any one of the three sub-criteria fails.
- How to match candidate relationships to expected ones by class pair (regardless of direction)
- How to handle extra/hallucinated relationships
  (present in candidate but not expected)
- How to handle missing relationships
  (expected but absent in candidate)
- How to handle relationships involving unmatched classes

Evaluation Steps:"""



Backup2="""You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the RELATIONSHIPS in a PlantUML candidate diagram against
a gold reference diagram. Cover:

- How to enumerate all relationships in the gold diagram,
  recording for each:
  (a) the two class endpoints
  (b) the relationship type
      (association, inheritance/generalization,
       composition, aggregation, dependency, realisation)
  (c) the multiplicity at each end
      (1, *, 0..1, 1..*, 0..*, etc.)

- How to enumerate all relationships in the candidate  diagram using the same format
- How to match candidate relationships to gold relationships by class pair
  (match by endpoints first, regardless of direction)
- How to measure Relationship Completeness (0-1):
  matched relationships / total gold relationships
- How to measure Relationship Correctness (0-1):
  correct relationships / total candidate relationships
  A relationship is CORRECT only if it is matched AND
  ALL THREE sub-criteria match the gold:
  1. TYPE matches
  2. MULTIPLICITY matches at both ends

  A relationship is INCORRECT if any one fails.
  Unmatched candidate relationships count as incorrect.
- How to handle naming variants in class names
  (e.g. OrderMgr vs OrderManager)
- How to handle extra/hallucinated relationships
- How to handle missing relationships

Evaluation Steps:"""

## Reflection

In [7]:
def needs_reflection(round_num, current_scores,previous_scores=None):
    """
    Decide whether to run a reflection round.

    Rules:
    - Round 1: always runs (previous_scores is None,
      no delta to compare yet)
    - Round 2: only runs if max score delta between
      initial scores and round 1 scores > DELTA
    - Never exceeds MAX_REFLECT rounds

    Args:
        round_num:       current round number (1-based)
        current_scores:  scores from most recent round
        previous_scores: scores from the round before
                         (None for round 1)
    """
    # Never exceed max reflection rounds
    if round_num > MAX_REFLECT:
        return False

    # Round 1 always runs — no previous scores to compare
    if previous_scores is None:
        return True

    # Round 2+ — only run if scores are still changing
    deltas = [
        abs(current_scores[k] - previous_scores[k])
        for k in current_scores
        if current_scores.get(k) is not None
        and previous_scores.get(k) is not None
    ]

    if not deltas:
        return False

    return max(deltas) > DELTA

## PROMPT

In [8]:
def s1_prompt(problem, candidate):
    """S1 — Simple baseline. No CoT, no gold."""
    return f"""You are evaluating the RELATIONSHIPS in a
PlantUML candidate diagram.

Definitions:
- Completeness (0-1): fraction of expected relationships
  present in the candidate diagram.
- Correctness (0-1): fraction of candidate relationships
  that are fully accurate.
  A relationship is correct only if Type (association, inheritance, composition,
     aggregation, dependency, realisation) match:



Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s2_prompt(problem, candidate, cot_steps):
    """
    S2 and S3 base scoring prompt.
    S3 uses this for initial scoring, then adds reflection.
    No gold reference.
    """
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s3_reflection_prompt(problem, candidate,
                                cot_steps, previous_scores,
                                round_num):
    """
    S3 — Reflection prompt. No gold reference.
    Round 1 always runs. Round 2 only if delta > DELTA.
    """
    comp = previous_scores['rel_completeness']
    corr = previous_scores['rel_correctness']

    return f"""You previously evaluated the RELATIONSHIPS in
this diagram (reflection round {round_num}) and produced:

- Relationship Completeness: {comp}
- Relationship Correctness:  {corr}

Please reflect carefully using ONLY the problem
description as your reference:

For Relationship Completeness:
1. Re-read the problem description carefully.
2. List every relationship you believe should exist
   between classes, based on the problem domain.
3. For each expected relationship, check if a
   corresponding one exists in the candidate diagram.
4. Recompute: completeness = found / expected
5. Revise your score if needed.

For Relationship Correctness:
1. List every relationship in the candidate diagram.
2. For each one, evaluate whether it is appropriate for this domain
3. Count: correct / total candidate relationships
4. Revise your score if needed.

Remember: a relationship is INCORRECT if any one of
type, or multiplicity is wrong.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s4_prompt(problem, gold, candidate):
    """S4 — Reference only. No CoT."""
    return f"""You are evaluating the RELATIONSHIPS in a
PlantUML candidate diagram against a gold reference.

Definitions:
- Completeness (0-1): fraction of gold relationships
  present in the candidate.
  Formula: matched / total gold relationships
- Correctness (0-1): fraction of candidate relationships
  that are fully accurate compared to gold.
  Formula: correct / total candidate relationships

A relationship is MATCHED if the same class pair exists
in both the candidate and the gold (match by endpoints).

A matched relationship is CORRECT only if TYPE matches (association, inheritance,
     composition, aggregation, dependency, realisation) with the gold

Unmatched candidate relationships (hallucinated) and
matched but incorrect relationships both count against
correctness.

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s5_prompt(problem, gold, candidate, cot_steps):
    """
    S5 and S6 base scoring prompt.
    S6 uses this for initial scoring, then adds reflection.
    With gold reference.
    """
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s6_reflection_prompt(problem, gold, candidate,
                                cot_steps, previous_scores,
                                round_num):
    """
    S6 — Reflection prompt. With gold reference.
    Round 1 always runs. Round 2 only if delta > DELTA.
    """
    comp = previous_scores['rel_completeness']
    corr = previous_scores['rel_correctness']

    return f"""You previously evaluated the RELATIONSHIPS in
this diagram (reflection round {round_num}) and produced:

- Relationship Completeness: {comp}
- Relationship Correctness:  {corr}

Please reflect carefully using the gold reference.

For Relationship Completeness:
1. List every relationship in the gold diagram with class pair.
2. For each gold relationship, check if a matching
   one exists in the candidate (match by class pair).
3. Count: matched / total gold relationships
4. Verify and revise your completeness score.

For Relationship Correctness:
1. List every relationship in the candidate diagram.
2. For each one, find its gold match (by class pair).
3. For each matched pair, verify TYPE match the gold type?

4. Mark as INCORRECT if it fails.
5. Also mark unmatched candidate relationships as
   INCORRECT (hallucinated relationships).
6. Count: correct / total candidate relationships
7. Verify and revise your correctness score.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


##SCORE PARSER

In [9]:
import re

def parse_rel_scores(response):
    """
    Extract relationship completeness and correctness
    from LLM response text.
    Clamps values to [0, 1].
    Returns None for any score that cannot be parsed.
    """
    scores = {
        'rel_completeness': None,
        'rel_correctness':  None
    }

    if response is None:
        return scores

    # Remove markdown bold markers before processing
    clean_response = response.replace("**", "")

    for line in clean_response.strip().split("\n"):
        line_lower = line.lower()

        # Find a number in the line (int or float, optional %)
        match = re.search(r'[\=\:]\s*([\d\.]+)\s*%?', line)
        if not match:
            continue

        try:
            value = float(match.group(1))

            # Handle percentage format (e.g. 83.3% → 0.833)
            if "%" in line:
                value = value / 100

            # Clamp to [0, 1]
            value = max(0.0, min(1.0, value))

            if "relationship completeness" in line_lower:
                scores['rel_completeness'] = value
            elif "relationship correctness" in line_lower:
                scores['rel_correctness'] = value

        except ValueError:
            continue

    return scores

In [10]:
import re

def parse_attr_scores(response):
    """
    Extract class completeness and correctness
    from LLM response text.
    """
    scores = {
        'rel_completeness': None,
        'rel_correctness':  None
    }

    print(response)
    print("Next###############")
    if response is None:
        return scores

    def extract_value(line):
        """
        Extract a numeric score from a single line.
        Handles decimal (0.8) and fraction (8/10).
        Returns float in [0, 1] or None.
        """
        # Try fraction first: 8/10
        fraction = re.search(r'(\d+)\s*/\s*(\d+)', line)
        if fraction:
            numerator   = float(fraction.group(1))
            denominator = float(fraction.group(2))
            if denominator > 0:
                return max(0.0, min(1.0,
                           numerator / denominator))
            return None

        # Try decimal or integer: 0.8 or 1 or 0
        decimal = re.search(r':\s*([0-1](?:\.\d+)?)', line)
        if decimal:
            return max(0.0, min(1.0,
                       float(decimal.group(1))))

        return None

    for line in response.strip().split('\n'):
        line_lower = line.lower()



    if "relationship completeness" in line_lower:
        scores['rel_completeness'] = extract_value(line)
    elif "relationship correctness" in line_lower:
        scores['rel_correctness'] = extract_value(line)


    return scores

##EVALUATOR

In [11]:
def evaluate_relationships(scenario, problem, candidate, model,
                           gold=None, cot_steps_no_gold=None,
                           cot_steps_with_gold=None):
    """
    Evaluate RELATIONSHIP dimension for one candidate
    under a given scenario.

    Returns dict:
        rel_completeness    : float or None
        rel_correctness     : float or None
        reflected           : bool
        reflection_rounds   : int
    """
    reflected         = False
    reflection_rounds = 0

    # ── S1 ──────────────────────────────────────────────
    if scenario == 's1':
        prompt   = s1_prompt(problem, candidate)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S2 ──────────────────────────────────────────────
    elif scenario == 's2':
        prompt   = s2_prompt(
                     problem, candidate, cot_steps_no_gold)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S3 — Auto-CoT + Reflection ──────────────────────
    # Uses same CoT steps as S2 (no gold)
    elif scenario == 's3':
        # Initial scoring — same as S2
        prompt          = s2_prompt(
                            problem, candidate, cot_steps_no_gold)
        response        = call_llm(prompt, model)
        scores          = parse_rel_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,
                                    previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s3_reflection_prompt(
                problem, candidate, cot_steps_no_gold,
                previous_scores, round_num
            )
            response = call_llm(ref_prompt, model)
            scores   = parse_rel_scores(response)

    # ── S4 ──────────────────────────────────────────────
    elif scenario == 's4':
        prompt   = s4_prompt(problem, gold, candidate)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S5 ──────────────────────────────────────────────
    elif scenario == 's5':
        prompt   = s5_prompt(
                     problem, gold, candidate,
                     cot_steps_with_gold)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S6 — Reference + CoT + Reflection ───────────────
    elif scenario == 's6':
        # Initial scoring — same as S5
        prompt          = s5_prompt(
                            problem, gold, candidate,
                            cot_steps_with_gold)
        response        = call_llm(prompt, model)
        scores          = parse_rel_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,
                                    previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s6_reflection_prompt(
                problem, gold, candidate, cot_steps_with_gold,
                previous_scores, round_num
            )
            response = call_llm(ref_prompt, model)
            scores   = parse_rel_scores(response)

    else:
        print(f"  Unknown scenario: {scenario}")
        scores = {
            'rel_completeness': None,
            'rel_correctness':  None
        }

    scores['reflected']         = reflected
    scores['reflection_rounds'] = reflection_rounds
    return scores


##DATA LOADING

In [12]:
import chardet

def load_gold(filepath):
    with open(filepath, 'rb') as f:
        detected = chardet.detect(f.read())
    encoding = detected['encoding'] or 'windows-1252'

    gold = {}
    with open(filepath, newline='', encoding=encoding) as f:
        reader = csv.DictReader(f)
        for row in reader:
            gold[row['id']] = {
                'problem':  row['problem'],
                'solution': row['solution']
            }
    return gold


def load_candidates(filepath):
    """
    Load candidates CSV.
    Columns: id, student_answer,
             score1 (class_completeness),
             score2 (class_correctness),
             score3 (attr_completeness),
             score4 (attr_correctness),
             score5 (rel_completeness),
             score6 (rel_correctness)

    id directly matches gold file id.
    """
    candidates = []
    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            candidates.append({
                'id':      row['id'],
                'diagram': row['student_answer'],
                'human_scores': {
                    'class_completeness': float(row['score1']),
                    'class_correctness':  float(row['score2']),
                    'attr_completeness':  float(row['score3']),
                    'attr_correctness':   float(row['score4']),
                    'rel_completeness':   float(row['score5']),
                    'rel_correctness':    float(row['score6'])
                }
            })
    return candidates



In [13]:
def save_results(results, output_filepath):
    """Save LLM scores + human scores to CSV."""
    fieldnames = [
        'candidate_id', 'scenario', 'model',
        'reflected', 'reflection_rounds',
        'human_rel_completeness', 'human_rel_correctness',
        'llm_rel_completeness',   'llm_rel_correctness'
    ]
    with open(output_filepath, 'w', newline='',
              encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"\n  Results saved to: {output_filepath}")

In [14]:
def run_relationship_evaluation(gold_file, candidate_file,
                                scenario, model, output_file):
    """
    Main loop — evaluates RELATIONSHIP dimension for all
    candidates under a given scenario and model.
    """
    print(f"\n{'='*55}")
    print(f"  CD-EVAL — Relationship Dimension")
    print(f"  Scenario        : {scenario.upper()}")
    print(f"  Model           : {model}")
    print(f"  Max reflections : {MAX_REFLECT}")
    print(f"  Delta threshold : {DELTA}")
    print(f"{'='*55}")

    # Load data
    print("\nLoading data...")
    gold       = load_gold(gold_file)
    candidates = load_candidates(candidate_file)
    print(f"  Gold scenarios : {len(gold)}")
    print(f"  Candidates     : {len(candidates)}")

    # Generate CoT steps once if needed
    # S2 and S3 share the same no-gold steps
    # S5 and S6 share the same with-gold steps
    cot_steps_no_gold   = None
    cot_steps_with_gold = None

    if scenario in ['s2', 's3']:
        cot_steps_no_gold = generate_cot_steps_no_gold(model)

    elif scenario in ['s5', 's6']:
        cot_steps_with_gold = generate_cot_steps_with_gold(model)

    # Evaluation loop
    results = []
    print(f"\nEvaluating {len(candidates)} candidates...\n")

    for candidate in tqdm(candidates):

        cid          = candidate['id']
        diagram      = candidate['diagram']
        human_scores = candidate['human_scores']

        # id directly matches gold
        if cid not in gold:
            print(f"\n  Warning: no gold found for "
                  f"id={cid} — skipping.")
            continue

        problem  = gold[cid]['problem']
        solution = gold[cid]['solution']

        # Run evaluation
        llm_scores = evaluate_relationships(
            scenario            = scenario,
            problem             = problem,
            candidate           = diagram,
            model               = model,
            gold                = solution,
            cot_steps_no_gold   = cot_steps_no_gold,
            cot_steps_with_gold = cot_steps_with_gold
        )

        # Collect result row
        results.append({
            'candidate_id':    cid,
            'scenario':        scenario,
            'model':           model,
            'reflected':       llm_scores['reflected'],
            'reflection_rounds': llm_scores['reflection_rounds'],
            # human scores (relationship dimension only)
            'human_rel_completeness':
                human_scores['rel_completeness'],
            'human_rel_correctness':
                human_scores['rel_correctness'],
            # llm scores
            'llm_rel_completeness':
                llm_scores['rel_completeness'],
            'llm_rel_correctness':
                llm_scores['rel_correctness']
        })

        time.sleep(0.5)  # avoid rate limits

    save_results(results, output_file)

    # Summary
    reflected_count = sum(1 for r in results if r['reflected'])
    print(f"\n  Total evaluated     : {len(results)}")
    print(f"  Reflection triggered: {reflected_count} "
          f"({100*reflected_count//max(len(results),1)}%)")

In [ ]:
gold_file = "GoldDataset.csv"
candidate_file= "CandidatesDataset.csv"
scenario= "s5"
model="gpt-5.4"
output_file="ClaudeRelRevisedS5.csv"

run_relationship_evaluation(gold_file, candidate_file,scenario, model, output_file)


In [16]:
if __name__ == "__main__":

    parser = argparse.ArgumentParser(
        description="CD-EVAL: Relationship dimension evaluator"
    )
    parser.add_argument(
        "--gold", required=True,
        help="Path to gold CSV (columns: id, problem, solution)"
    )
    parser.add_argument(
        "--candidates", required=True,
        help="Path to candidates CSV"
    )
    parser.add_argument(
        "--scenario", required=True,
        choices=['s1', 's2', 's3', 's4', 's5', 's6'],
        help="Evaluation scenario"
    )
    parser.add_argument(
        "--model", default="gpt-4",
        choices=list(MODELS.keys()),
        help="LLM to use as evaluator"
    )
    parser.add_argument(
        "--output", required=True,
        help="Path to output CSV file"
    )

    args = parser.parse_args()

    run_relationship_evaluation(
        gold_file      = args.gold,
        candidate_file = args.candidates,
        scenario       = args.scenario,
        model          = args.model,
        output_file    = args.output
    )


usage: colab_kernel_launcher.py [-h] --gold GOLD --candidates CANDIDATES
                                --scenario {s1,s2,s3,s4,s5,s6}
                                [--model {gpt-5.4,gpt-5.4-mini,claude-opus-4-6,claude-sonnet-4-5}]
                                --output OUTPUT
colab_kernel_launcher.py: error: the following arguments are required: --gold, --candidates, --scenario, --output
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --gold, --candidates, --scenario, --output

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_14386/2147556516.py", line 33, in <cell line: 0>
    args = parser.parse_args()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1904, in parse_args
    args, argv = se

TypeError: object of type 'NoneType' has no len()